In [0]:
from pyspark.sql.functions import current_timestamp, col

source_path = (
    "/Volumes/fintech_lakehouse/bronze/raw_events/orders/"
)

schema_path = (
    "/Volumes/fintech_lakehouse/bronze/raw_events/"
    "_schemas/orders/"
)

checkpoint_path = (
    "/Volumes/fintech_lakehouse/bronze/raw_events/"
    "_checkpoints/bronze_orders/"
)

bronze_table = "fintech_lakehouse.bronze.orders_raw"

bronze_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("rescuedDataColumn", "_rescued_data")
    .load(source_path)
    .withColumn("ingested_at", current_timestamp())
    .withColumn("source_file", col("_metadata.file_path"))
    .withColumn(
        "source_modified_at",
        col("_metadata.file_modification_time")
    )
)

query = (
    bronze_df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(bronze_table)
)

query.awaitTermination()

print(f"Bronze ingestion completed: {bronze_table}")

display(
    spark.table(bronze_table)
    .orderBy(col("ingested_at").desc())
)